# Membryo training

This public notebook was migrated from the audited read-only research source. Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'Membryo'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import re
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_2d as s1
from stvirtual.models import stage2_2d as s2


In [ ]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)


In [ ]:
route_ids=['E9.5', 'E11.5', 'E13.5', 'E15.5', 'E16.5']
fracs = ['E9.5_to_E11.5','E11.5_to_E13.5','E13.5_to_E15.5', 'E15.5_to_E16.5']
seg_key = "E15.5_to_E16.5" 
steps = 20


In [ ]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


In [ ]:
adata.obs['timepoint'].unique()


In [ ]:
model = None


In [ ]:
importlib.reload(s1)

res1 = s1.train_model_multislice(
    model=model,                  
    adata_all=adata,
    slice_key="timepoint",
    route_ids=route_ids,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    latent_key="X_scanVI",    
    steps=steps,  
    guide_eps=0.01,
    uot_eps=0.05, uot_tau=0.5, uot_lam_x=2.0, uot_lam_f=0.05,
    guide_topk=512, guide_temp=0.5, guide_schedule="linear",
    terrain_gap=False,
    cell_type_key='annotation',
    lam_context=10.0,
    lam_residual=1.0,
    lam_vsmooth=0.1,  
    lam_uot=1.0,
    lib_layer='count',              
    latent_dim=10,             
    epochs=100,
    device="cuda:0",
)


## Save Stage-1 traces and generate 2D boundaries

In [ ]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import rollout_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="timepoint", ann_key="annotation",
    save_prefix="rollout_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = rollout_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


In [ ]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=SCANVI_ADATA,
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "E9.5_to_E11.5" / "checkpoints" / "best.pt"),
    device=device,
    layer_col="annotation",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key="X_scanVI",
            z_csv_offset=0,
            layer_col="annotation",  
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'timepoint',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=100, LR=1e-4),
    rl_xy=2.0,
    rl_z=0.05,
)

for o in outs:
    print(o["best_ckpt_path"])   


In [ ]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


In [ ]:
importlib.reload(s2)
rollouts = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    rollouts[seg_key] = s2.rollout_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],sample_key='timepoint',
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'rollout' / seg_key, output_prefix=seg_key,  
    )
    print(seg_key, "Tp1=", len(rollouts[seg_key]["coords"]))


## Persist rollout identity metadata in each H5AD frame

This non-lineage model writes only `uid` and `parent_uid` into `adata.obs`. No differentiation fields are created. Existing notebook outputs are preserved.


In [ ]:
ROLLOUT_OBS_FIELDS = ("uid", "parent_uid")

def persist_rollout_obs(rollout_result):
    output_paths = rollout_result.get("output_paths")
    if not isinstance(output_paths, list):
        raise KeyError("rollout_result has no output_paths; pass output_dir to rollout_policy_one_stage")
    if len(output_paths) != len(rollout_result["coords"]):
        raise ValueError("output_paths and rollout frames have different lengths")

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        for field in ROLLOUT_OBS_FIELDS:
            frames = rollout_result.get(field)
            if not isinstance(frames, list):
                raise KeyError(f"rollout_result does not provide {field!r}")
            values = np.asarray(frames[frame_index])
            if values.ndim != 1 or len(values) != frame_adata.n_obs:
                raise ValueError(
                    f"{field} at frame {frame_index} has shape {values.shape}; "
                    f"expected ({frame_adata.n_obs},)"
                )
            frame_adata.obs[field] = values
        frame_adata.obs_names = frame_adata.obs["uid"].astype(str).to_numpy()
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": len(output_paths), "obs_fields": list(ROLLOUT_OBS_FIELDS)}

rollout_obs_reports = {
    route: persist_rollout_obs(route_rollout)
    for route, route_rollout in rollouts.items()
}
rollout_obs_reports


In [ ]:
ro = rollouts[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


In [ ]:
ad1 = sc.read_h5ad(str(RUN_ROOT / 'rollout_h5ad_mosta' / 'E11.5_to_E13.5_f0010_t1p0000.h5ad'))
ad1.obs['layer_name'].value_counts()


In [ ]:
adata135 = adata[adata.obs['timepoint']=='E13.5']
adata135.obs['annotation'].value_counts()


In [ ]:
from matplotlib.path import Path as MplPath
import numpy as np
import matplotlib.pyplot as plt
import os

def _pick_shell(pack, prefer="T"):
    # prefer="T":  pack.T  shell；； None
    if getattr(pack, "shells_norm", None) is None or len(pack.shells_norm) == 0:
        return None
    if prefer == "T" and (0 <= int(pack.T) < len(pack.shells_norm)):
        return pack.shells_norm[int(pack.T)]
    return pack.shells_norm[-1]

def plot_each_stage_last_frame(
    s2, ctx, stages,
    *, sample_key="timepoint", layer_col="annotation",
    out_dir="debug_stage_last",
    overlay_target=True,
    shell_prefer="T",          # "T" or "last"
    s_last=1, s_out=4,
    alpha_in=0.25, alpha_out=0.85,
    dpi=100,
    show=False,
):
    os.makedirs(out_dir, exist_ok=True)

    for si, cfg in enumerate(stages):
        pack = s2.prepare_one_stage(ctx, cfg, sample_key=sample_key, layer_col=layer_col)

        # last frame from npz / coords_frames
        xy_last = pack.coords_seq_torch[-1].detach().cpu().numpy()
        shell = _pick_shell(pack, prefer=shell_prefer)

        if shell is not None:
            poly = MplPath(shell)
            inside_last = poly.contains_points(xy_last)
        else:
            inside_last = np.ones((xy_last.shape[0],), dtype=bool)

        title = f"stage{si:02d} {getattr(cfg,'src','?')} -> {getattr(cfg,'tgt','?')} | last_frame"
        plt.figure(figsize=(6, 6), dpi=dpi)

        # last frame: inside vs outside shell
        plt.scatter(xy_last[inside_last, 0], xy_last[inside_last, 1],
                    s=s_last, alpha=alpha_in, label="last(in_shell)")
        plt.scatter(xy_last[~inside_last, 0], xy_last[~inside_last, 1],
                    s=s_out, alpha=alpha_out, label="last(out_shell)")

        # optional: overlay target
        if overlay_target:
            xy_tgt = pack.target_state["coords"].detach().cpu().numpy()
            if shell is not None:
                inside_tgt = poly.contains_points(xy_tgt)
            else:
                inside_tgt = np.ones((xy_tgt.shape[0],), dtype=bool)

            plt.scatter(xy_tgt[inside_tgt, 0], xy_tgt[inside_tgt, 1],
                        s=1, alpha=0.10, label="tgt(in_shell)")
            plt.scatter(xy_tgt[~inside_tgt, 0], xy_tgt[~inside_tgt, 1],
                        s=6, alpha=0.20, label="tgt(out_shell)")

        # boundary
        if shell is not None:
            plt.plot(np.r_[shell[:, 0], shell[0, 0]],
                     np.r_[shell[:, 1], shell[0, 1]],
                     lw=2)

        plt.title(title)
        plt.axis("equal")
        plt.tight_layout()
        plt.legend(markerscale=4, frameon=False, loc="upper right")

        out_path = os.path.join(out_dir, f"{si:02d}_{getattr(cfg,'src','src')}_to_{getattr(cfg,'tgt','tgt')}_last.png")
        plt.savefig(out_path, bbox_inches="tight")
        if show:
            plt.show()
        else:
            plt.close()

        print("[saved]", out_path)

# 
plot_each_stage_last_frame(
    s2, ctx, stages,
    sample_key="timepoint",
    layer_col="annotation",
    out_dir="debug_stage_last",
    overlay_target=True,
    shell_prefer="T",
    show=True,
)


In [ ]:
CKPT_3DSLICE = str(STAGE1_CHECKPOINT_ROOT / 'E9.5_to_E11.5' / 'checkpoints' / 'best.pt')  #  ckpt['global_norm']['xy_mu'], ['xy_s']

SPATIAL_KEY = "spatial_aligned_centroid"            # .obsm 
CELL_TYPE_KEY = "annotation"    # .obs 
IMG_SIZE = 32
# =======================================================


# =========================
# Norm utils (only for TRUE)
# =========================
def get_xy_norm_from_ckpt(ckpt_obj: dict):
    """Read xy_mu, xy_s from 3dslice checkpoint."""
    norm = ckpt_obj.get("global_norm", None)
    if norm is None:
        raise KeyError("Checkpoint has no key 'global_norm'")

    def _to_np(x):
        if torch.is_tensor(x):
            x = x.detach().cpu().numpy()
        return np.asarray(x, dtype=np.float32)

    xy_mu = _to_np(norm["xy_mu"]).reshape(1, -1)

    xy_s = _to_np(norm["xy_s"])
    if xy_s.ndim == 0:
        xy_s = np.array([[xy_s]], dtype=np.float32)
    elif xy_s.ndim == 1:
        xy_s = xy_s.reshape(1, -1)
    else:
        xy_s = xy_s.reshape(1, -1)

    return xy_mu.astype(np.float32), xy_s.astype(np.float32)


def normalize_coords(coords: np.ndarray, xy_mu: np.ndarray, xy_s: np.ndarray):
    """(N,2) -> normalized using ctx.xy_mu, ctx.xy_s."""
    coords = np.asarray(coords, dtype=np.float32)
    if coords.ndim != 2 or coords.shape[1] < 2:
        raise ValueError(f"coords must be (N,2+) array, got {coords.shape}")
    xy = coords[:, :2].astype(np.float32)
    return (xy - xy_mu) / (xy_s + 1e-12)


# =========================
# Raster + Dice
# =========================
def get_common_metadata(adata_pred, adata_true):
    """
    ：
    1)  bounds
    2)  type_to_id（0）
    """
    coords_p = np.asarray(adata_pred.obsm[SPATIAL_KEY], dtype=np.float32)
    coords_t = np.asarray(adata_true.obsm[SPATIAL_KEY], dtype=np.float32)

    all_coords = np.vstack([coords_p, coords_t])
    x_min, x_max = all_coords[:, 0].min(), all_coords[:, 0].max()
    y_min, y_max = all_coords[:, 1].min(), all_coords[:, 1].max()

    pad_x = (x_max - x_min) * 0.05
    pad_y = (y_max - y_min) * 0.05
    bounds = (x_min - pad_x, x_max + pad_x, y_min - pad_y, y_max + pad_y)

    types_p = set(adata_pred.obs[CELL_TYPE_KEY].astype(str).unique())
    types_t = set(adata_true.obs[CELL_TYPE_KEY].astype(str).unique())
    all_types = sorted(list(types_p | types_t))

    type_to_id = {t: i + 1 for i, t in enumerate(all_types)}
    return bounds, type_to_id, all_types


def rasterize_h5ad(adata, bounds, type_to_id, img_size):
    coords = np.asarray(adata.obsm[SPATIAL_KEY], dtype=np.float32)
    cell_labels = adata.obs[CELL_TYPE_KEY].astype(str)

    img = np.zeros((img_size, img_size), dtype=np.uint8)
    x_min, x_max, y_min, y_max = bounds

    norm_x = (coords[:, 0] - x_min) / (x_max - x_min + 1e-12)
    norm_y = (coords[:, 1] - y_min) / (y_max - y_min + 1e-12)

    px = (norm_x * (img_size - 1)).astype(np.int64)
    py = (img_size - 1) - (norm_y * (img_size - 1)).astype(np.int64)

    valid = (px >= 0) & (px < img_size) & (py >= 0) & (py < img_size)
    px, py = px[valid], py[valid]
    valid_labels = cell_labels.iloc[valid]

    ids = valid_labels.map(type_to_id).fillna(0).astype(np.int64).values
    img[py, px] = ids
    return img


def calculate_dice(mask_pred, mask_true):
    intersection = (mask_pred & mask_true).sum()
    sum_area = mask_pred.sum() + mask_true.sum()
    if sum_area == 0:
        return 1.0
    return 2.0 * intersection / sum_area


In [ ]:
import matplotlib.pyplot as plt
adata_pred = sc.read_h5ad(str(RUN_ROOT / 'rollout_h5ad_mosta' / 'E11.5_to_E13.5_f0010_t0p5000.h5ad'))
adata_true = adata[adata.obs['timepoint']=='E12.5'].copy()
adata_pred.obs['annotation'] = adata_pred.obs['layer_name']

ad_pred = adata_pred
ad_true = adata_true

ckpt = torch.load(CKPT_3DSLICE, map_location="cpu")
xy_mu, xy_s = get_xy_norm_from_ckpt(ckpt)

true_xy = ad_true.obsm[SPATIAL_KEY]
ad_true.obsm[SPATIAL_KEY] = normalize_coords(true_xy, xy_mu, xy_s)
ad_pred.obsm[SPATIAL_KEY] = ad_pred.obsm['spatial']  # 
bounds, type_to_id, all_types = get_common_metadata(ad_pred, ad_true)

img_pred = rasterize_h5ad(ad_pred, bounds, type_to_id, IMG_SIZE)
img_true = rasterize_h5ad(ad_true, bounds, type_to_id, IMG_SIZE)

dice_scores = {}
for cell_name in all_types:
    cid = type_to_id[cell_name]
    mask_p = (img_pred == cid)
    mask_t = (img_true == cid)
    score = calculate_dice(mask_p, mask_t)
    dice_scores[cell_name] = score

avg_dice = float(np.mean(list(dice_scores.values()))) if len(dice_scores) else 0.0
print(f"\n>>> Mean Dice (macro): {avg_dice:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
cmap = "jet"
max_id = len(all_types) + 1

axes[0].imshow(img_true, cmap=cmap, vmin=0, vmax=max_id, interpolation="nearest")
axes[0].set_title("Ground Truth (TRUE normalized)")
axes[0].axis("off")

axes[1].imshow(img_pred, cmap=cmap, vmin=0, vmax=max_id, interpolation="nearest")
axes[1].set_title(f"Prediction (PRED raw)\nMean Dice: {avg_dice:.3f}")
axes[1].axis("off")

plt.tight_layout()
plt.show()
